In [1]:
"""
Cell 1: Loading Both Datasets
We need our base RFM/Churn dataset to add columns to, and we need the raw transactions to calculate the new features.
"""

# Cell 1
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# Define paths
CLEANED_DATA_PATH = '../data/processed/cleaned_online_retail.parquet'
BASE_FEATURES_PATH = '../data/processed/rfm_and_churn_labels.parquet'
ADVANCED_FEATURES_PATH = '../data/processed/advanced_features_and_churn.parquet'

print("Loading datasets...")
df_raw = pd.read_parquet(CLEANED_DATA_PATH)
df_base = pd.read_parquet(BASE_FEATURES_PATH)

print(f"Raw Transactions Shape: {df_raw.shape}")
print(f"Base Features Shape: {df_base.shape}")

Loading datasets...
Raw Transactions Shape: (397885, 10)
Base Features Shape: (3370, 5)


In [2]:
"""
Cell 2: Re-enforcing the Time Wall
We dynamically recreate the exact same 90-day cutoff we used in Phase 2. We will only use the observation_data to engineer our new features.
"""
# Cell 2
# Recreate the exact same cutoff date from Phase 2
max_date = df_raw['Date'].max()
performance_days = 90
cutoff_date = max_date - pd.Timedelta(days=performance_days)

# Isolate the past (Observation Window)
observation_data = df_raw[df_raw['Date'] < cutoff_date].copy()

print(f"Cutoff Date locked at: {cutoff_date.date()}")
print(f"Transactions available for feature engineering: {len(observation_data)}")

Cutoff Date locked at: 2011-09-10
Transactions available for feature engineering: 236358


In [3]:
"""
Cell 3: Engineering Tenure and Purchase Velocity
Tenure is how long they've been a customer. Velocity is how frequently they buy.
"""

# Cell 3
print("Calculating Tenure and Velocity...")

# Find the very first purchase date for each customer in the observation window
first_purchase = observation_data.groupby('CustomerID')['Date'].min().reset_index()
first_purchase.rename(columns={'Date': 'FirstPurchaseDate'}, inplace=True)

# Calculate Tenure (Days between their first purchase and the cutoff date)
first_purchase['Tenure'] = (cutoff_date - first_purchase['FirstPurchaseDate']).dt.days

# Merge Tenure into our base feature set
df_advanced = pd.merge(df_base, first_purchase[['CustomerID', 'Tenure']], on='CustomerID', how='left')

# Calculate Purchase Velocity (Average days between orders)
# If they only bought once (Frequency == 1), their velocity is their entire tenure.
df_advanced['Velocity'] = np.where(
    df_advanced['Frequency'] > 1, 
    df_advanced['Tenure'] / df_advanced['Frequency'], 
    df_advanced['Tenure']
)

display(df_advanced[['CustomerID', 'Frequency', 'Recency', 'Tenure', 'Velocity']].head())

Calculating Tenure and Velocity...


,CustomerID,Frequency,Recency,Tenure,Velocity
0,12346,1,235,235,235.000000
1,12347,5,39,277,55.400000
2,12348,3,158,268,89.333333
3,12350,1,220,220,220.000000
4,12352,5,172,206,41.200000


In [4]:
"""
Cell 4: Engineering Financial Context and Item Diversity
We calculate the Average Order Value (AOV) and count how many unique items they rely on your store for.
"""
# Cell 4
print("Calculating Financial Context and Diversity...")

# 1. Average Order Value (AOV)
df_advanced['AOV'] = df_advanced['Monetary'] / df_advanced['Frequency']

# 2. Item Diversity (Number of unique products bought over their lifetime)
diversity = observation_data.groupby('CustomerID')['StockCode'].nunique().reset_index()
diversity.rename(columns={'StockCode': 'ItemDiversity'}, inplace=True)

# Merge Diversity into the dataset
df_advanced = pd.merge(df_advanced, diversity, on='CustomerID', how='left')

display(df_advanced[['CustomerID', 'Monetary', 'AOV', 'ItemDiversity']].head())


Calculating Financial Context and Diversity...


,CustomerID,Monetary,AOV,ItemDiversity
0,12346,77183.60,77183.600000,1
1,12347,2790.86,558.172000,82
2,12348,1487.24,495.746667,22
3,12350,334.40,334.400000,17
4,12352,1561.81,312.362000,26


In [5]:
"""
Cell 5: Final Clean-up and Assertions
We check for any weird math errors (like dividing by zero creating infinite values) and run our pipeline tests.
"""
# Cell 5
# Fill any potential NaNs that might have sneaked in from merging
df_advanced.fillna(0, inplace=True)

# Ensure no infinite values from division
df_advanced.replace([np.inf, -np.inf], 0, inplace=True)

# --- PRODUCTION PIPELINE CHECKS ---
assert df_advanced['Tenure'].min() >= 0, "Pipeline Error: Negative Tenure detected!"
assert df_advanced['AOV'].min() >= 0, "Pipeline Error: Negative AOV detected!"
assert df_advanced['Velocity'].isnull().sum() == 0, "Pipeline Error: Nulls in Velocity!"
assert len(df_advanced) == len(df_base), "Pipeline Error: We lost or duplicated customers during merging!"

print("Advanced Features engineered successfully. All assertions passed.")
# Let's look at the final feature set for the ML models
display(df_advanced.head())


Advanced Features engineered successfully. All assertions passed.


,CustomerID,Recency,Frequency,Monetary,Churn,Tenure,Velocity,AOV,ItemDiversity
0,12346,235,1,77183.60,1,235,235.000000,77183.600000,1
1,12347,39,5,2790.86,0,277,55.400000,558.172000,82
2,12348,158,3,1487.24,0,268,89.333333,495.746667,22
3,12350,220,1,334.40,1,220,220.000000,334.400000,17
4,12352,172,5,1561.81,0,206,41.200000,312.362000,26


In [ ]:
"""
Cell 6: Saving the Final Feature Matrix
This is the "gold mine" dataset we will feed into XGBoost.
"""

# Cell 6
# Save to processed folder
df_advanced.to_parquet(ADVANCED_FEATURES_PATH, index=False)
print(f"Advanced Modeling dataset successfully saved to: {ADVANCED_FEATURES_PATH}")

Advanced Modeling dataset successfully saved to: ../data/processed/advanced_features_and_churn.parquet


: 